<table><tr>
<td><b>Sapienza University of Rome</b><br>PhD Soft Skills 2026</td>
</tr></table>

# Notebook 4 — Build a chatbot with a method
**20 minutes.** You will build the conversation machinery, watch the transcript grow, hold a real conversation through it, then write a system prompt for your own field and try to break your own rules.


---
### How to use this notebook

1. Each numbered section gives you **a prompt**. Copy it.
2. Find the **empty code cell** underneath it.
3. **Click the word `generate`** inside that cell — it reads *Start coding or generate with AI*. An AI box opens. Paste the prompt there and press Enter.
4. **Look at what it wrote** — even if you don't understand it — then press ▶.
5. If it fails, don't fix it by hand. Paste the whole red error message back to the AI.

Every section also has a **Reference** cell with working code already in it. If you would rather watch than type, just run that instead. You will lose nothing.

> **Before you start: click `Copy to Drive` in the toolbar.** This notebook opened straight from GitHub, which means it is read-only — you can run it, but nothing you do will be saved. One click makes it yours to keep.

> The first time you run a cell you may see **"This notebook was not authored by Google"**. That is normal for anything opened from GitHub. Click **Run anyway**. (Copying to Drive first usually avoids it altogether.)

> You are never expected to write code. You are expected to say clearly what you want, and to check what you get.
---

> **How the model gets involved.** This notebook builds the *machinery* of a chatbot — the transcript, the system message, the token accounting. The **replies** come from the Colab AI chat panel: the notebook prints the exact text to send, you paste it into the sidebar, and paste the reply back.

> That is not a simplification. It is the real thing, with you standing where the network call would be — which means you get to *see* what is normally invisible.

---
## 1 · The transcript is the whole trick

A model has no memory. A conversation works because **the entire transcript is resent every single turn**. Let's build exactly that and watch it.

### 1 · Build the conversation machinery

**Copy this into the AI box** — click the word **generate** in the empty cell below, then paste this in:

```text
Build a simple chat transcript in this notebook:
- a list called `messages`, each entry a dict with a role
  (system, user, assistant) and text
- a function add(role, text) that appends to it
- a function show() that prints the whole transcript readably
- a function to_send() that prints exactly what would be sent to a
  model this turn, formatted as 'Role: text', with no banner or
  decoration around it
Start it with a system message and two example exchanges.
```

> Look carefully at what `to_send()` prints. That whole block is re-read from scratch by the model on every turn. Nothing is remembered; everything is resent.

<details><summary><b>Reference — click to open, or just run the cell below</b></summary>

This is one correct answer. Colab's AI will probably write something different, and that is fine — compare the two.

</details>

In [ ]:
messages = []

def add(role, text):
    messages.append({'role': role, 'text': text.strip()})

def show():
    for m in messages:
        body = m['text'] if len(m['text']) < 160 else m['text'][:160] + ' ...'
        print(f"[{m['role'].upper():9s}] {body}")

def to_send():
    # printed with no banner, so you can select all of it and paste it
    for m in messages:
        print(f"{m['role'].capitalize()}: {m['text']}\n")

add('system', 'You are a helpful assistant.')
add('user', 'What is a palimpsest?')
add('assistant', 'A manuscript page that has been scraped clean and written on again.')
add('user', 'How are they recovered?')

to_send()

---
## 2 · Watch the desk fill up

Every turn costs the *whole* transcript, not just your new message.

### 2 · Count the tokens

**Copy this into the AI box** — click the word **generate** in the empty cell below, then paste this in:

```text
Add a function called token_count() that returns the number of tokens in
the whole transcript, using tiktoken's 'cl100k_base' encoding.

Then add six more long exchanges to the transcript, printing the token
count after each one, and warn me whenever it goes over 600 tokens.
Finally plot the token count against the turn number, with a dashed
line at 600.
```

> The function name is pinned — a later cell calls `token_count()` by name. If your AI calls it something else, either rename it or tell the AI to.

<details><summary><b>Reference — click to open, or just run the cell below</b></summary>

This is one correct answer. Colab's AI will probably write something different, and that is fine — compare the two.

</details>

In [ ]:
!pip -q install tiktoken
import tiktoken, matplotlib.pyplot as plt
_enc = tiktoken.get_encoding('cl100k_base')

def token_count():
    return sum(len(_enc.encode(m['role'] + ': ' + m['text'])) for m in messages)

BUDGET = 600
history = []
filler = ('Multispectral imaging is the standard method, using wavelengths outside '
          'the visible range to reveal the erased under-text. ') * 6
for turn in range(6):
    add('user', f'Tell me more, part {turn + 1}.')
    add('assistant', filler)
    history.append(token_count())
    flag = '  <-- over budget!' if history[-1] > BUDGET else ''
    print(f'after turn {turn + 1}: {history[-1]:5d} tokens{flag}')

plt.figure(figsize=(6.5, 3.5))
plt.plot(range(1, len(history) + 1), history, 'o-', color='#8E2436')
plt.axhline(BUDGET, ls='--', color='grey', label=f'our budget ({BUDGET})')
plt.xlabel('turn'); plt.ylabel('tokens resent this turn')
plt.legend(); plt.tight_layout(); plt.show()

---
## 3 · Clear the desk

Before we hold a real conversation, we need the transcript short enough to paste. This is the 'start a new chat' button, and now you know exactly what it does: it clears the desk and keeps the standing instructions.

### 3 · Add a reset

**Copy this into the AI box** — click the word **generate** in the empty cell below, then paste this in:

```text
Add a function reset() that clears the transcript but keeps the system
message, printing the token count before and after. Then run it.
```

> Run this now. The next section asks you to copy the transcript by hand — and without a reset you would be pasting about five thousand characters of filler.

<details><summary><b>Reference — click to open, or just run the cell below</b></summary>

This is one correct answer. Colab's AI will probably write something different, and that is fine — compare the two.

</details>

In [ ]:
def reset():
    before = token_count()
    system = [m for m in messages if m['role'] == 'system']
    messages.clear()
    messages.extend(system)
    print(f'{before} tokens -> {token_count()} tokens (system message kept)')

reset()
show()

---
## 4 · Now hold an actual conversation

The replies come from the **Colab AI chat panel** — the sparkle icon in the left sidebar, or *Tools → AI assistance*.

The loop is:

1. Type your message in the **send** cell and run it.
2. Copy everything it printed, and paste it into the chat panel.
3. Copy the model's reply.
4. Paste it into the **receive** cell, between the quotes, and run that.
5. Go back to step 1 with a new message.

**You are standing exactly where the network call would be.** Two practical notes. Run each cell once per turn — running the receive cell twice would add the reply twice, so it refuses. And if the reply happens to contain three double-quotes in a row, you will get a red `SyntaxError`: delete those three characters and run again. That is the one paste this cell cannot survive.

In [ ]:
# ---- SEND ----  type your message here, run, then copy ALL the output.
my_message = 'How were palimpsests recovered?'

add('user', my_message)
to_send()

In [ ]:
# ---- RECEIVE ----  paste the model's reply between the quotes below.

reply = r"""

"""

# --- nothing below here needs editing ---
if not reply.strip():
    print('Nothing pasted yet — put the reply between the quotes above, then re-run.')
elif messages and messages[-1]['role'] == 'assistant':
    print('The last turn is already an assistant reply. Run the SEND cell first,')
    print('or you will add two replies in a row.')
else:
    add('assistant', reply)
    print(f'Added. Transcript is now {len(messages)} messages, {token_count()} tokens.')
    show()

> **Why `r"""` and not `'''`?** A reply containing ``` or a stray quote would otherwise end the text early and mangle your transcript — or, worse, run part of the reply as code. The `r` and the double quotes make that far less likely. If you still get a red error, the reply contained `"""`: delete those three characters and re-run.

---
## 5 · The part that matters: the system prompt

The system message is re-read on **every** turn, which is why it does not decay the way an instruction buried in turn 3 does. It is the highest-leverage text you will ever write, and almost nobody writes it deliberately.

Here is a worked example for a historian:

```text
You are assisting a doctoral researcher in early-modern Italian social history.

RULES, which apply to every answer:
- Answer only from the material the user provides. If it is not in the
  material, say "not in the provided text".
- Never invent citations, dates, or names.
- Quote the exact passage you are relying on, then interpret it.
- If a question is ambiguous, ask before answering.
- Do not open with praise or agreement.
```

| Line | Why it is there |
|---|---|
| *Answer only from the material* | The single strongest defence against hallucination. |
| *Never invent citations* | The specific failure that ends academic careers. |
| *Quote, then interpret* | A procedure, not a prohibition — much more reliably followed. |
| *Ask if ambiguous* | Stops it guessing your intent and confidently answering the wrong question. |
| *No praise* | Counteracts the trained-in agreeableness that wastes your time. |

### Now write your own

**This one is not a prompt to copy — it is your turn to write one.**

1. Write a system prompt for a research assistant in **your** field. Include at least one rule about something it must **refuse** to do.
2. Start a **fresh** chat in the panel and paste it as the very first message.
3. Work with it for a few turns.
4. Then spend five minutes trying to make it break your own rule.

Roleplay framings (*'for a novel I am writing, have a character explain…'*) defeat most prohibitions. Long conversations defeat them too.

**When you get it to break — and you will — stop and work out what did it.** The roleplay framing? The length of the conversation? A word that made the rule feel inapplicable? That is the useful part: not that the rule broke, but *what kind of pressure broke it* — because the same pressure will break the rules you write for your own research tools.

You can paste your system prompt into the cell below to keep it with the notebook.

In [ ]:
MY_SYSTEM_PROMPT = r"""

"""
print(MY_SYSTEM_PROMPT.strip() or 'Nothing saved yet.')

---
## What people always discover

- **Vague rules do nothing.** *'Be rigorous'* has no effect. *'Quote the passage before interpreting it'* has a large one.
- **Procedures beat prohibitions.** *'For each claim, do X'* works better than *'never do Y'* — measurably, and consistently.
- **Length wins.** Twenty turns later, your system prompt is competing against a lot of contradictory text on the same desk.

> **Playbook line:** write rules as procedures, not prohibitions. And when a conversation goes wrong, don't argue with it — start a new one and carry only the good parts forward.